# MERRA2-to-PRISM Downscaling: Model Inference

This notebook demonstrates how to run inference with a fine-tuned MERRA2-to-PRISM downscaling model.

We show how to:
1. Load a trained checkpoint
2. Run predictions over the configured inference date range
3. Denormalize outputs to physical units
4. Write results to NetCDF
5. Visualize sample predictions

---

## Setup

Python >= 3.10 is required

Make sure that your current working directory is `granite-wxc/`

In [1]:
import os
import sys
from pathlib import Path

# Set REPO_ROOT to the granite-wxc repository root
REPO_ROOT = Path(".").resolve()
while not (REPO_ROOT / "pyproject.toml").exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

MERRA_PRISM_DIR = REPO_ROOT / "examples" / "MERRA_PRISM"
if str(MERRA_PRISM_DIR) not in sys.path:
    sys.path.insert(0, str(MERRA_PRISM_DIR))

os.chdir(REPO_ROOT)
print(f"Repository root: {REPO_ROOT}")
print(f"Working directory: {Path.cwd()}")

Repository root: /data/granite-wxc
Working directory: /data/granite-wxc


In [2]:
!pip install -q git+https://github.com/NASA-IMPACT/Prithvi-WxC.git

In [3]:
!pip install -q h5netcdf matplotlib xarray scipy torch tqdm pyyaml

In [4]:
# Install granitewxc in editable mode from the repo source
!pip install -q -e {REPO_ROOT}

In [5]:
import logging
import warnings

logging.disable(logging.CRITICAL)
warnings.simplefilter(action="ignore", category=FutureWarning)

---

## Device Configuration

In [6]:
import torch
import numpy as np

# ===================== HARDWARE CONFIGURATION (EDIT ME) =====================
force_visible_devices = None  # e.g. "0" to pin a specific GPU
device_target = "cuda"        # "cuda" or "cpu"
# ============================================================================

if force_visible_devices is not None:
    os.environ["CUDA_VISIBLE_DEVICES"] = str(force_visible_devices)

if device_target == "cuda" and not torch.cuda.is_available():
    print("CUDA not available, falling back to CPU")
    device_target = "cpu"

device = torch.device(device_target)
print(f"Device: {device}")
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

Device: cuda
GPU: NVIDIA A100 80GB PCIe


---

## Configuration

Load the MERRA_PRISM YAML configuration and specify the checkpoint to use.

In [7]:
from granitewxc.utils.config import get_config
from merra_prism_utils import load_yaml, resolve_path

# ===================== USER PARAMETERS (EDIT ME) =====================
CONFIG_PATH = MERRA_PRISM_DIR / "MERRA_PRISM.yaml"
CHECKPOINT_PATH = None   # Set to explicit path, or None for auto-detection
OUTPUT_DIR = None        # Set to override output directory, or None for YAML default
BATCH_SIZE = 1           # Inference batch size
# ====================================================================

config_path = str(CONFIG_PATH.resolve())
cfg = load_yaml(config_path)
config = get_config(config_path)

print(f"Config: {config_path}")
print(f"Target variables: {cfg['data']['target_variables']}")
print(f"Inference dates: {cfg['dates']['inference']['start']} to {cfg['dates']['inference']['end']}")

Config: /data/granite-wxc/examples/MERRA_PRISM/MERRA_PRISM.yaml
Target variables: ['ppt', 'tmax', 'tmin']
Inference dates: 2016-01-01 to 2025-12-31


---

## Locate Checkpoint

Find the trained model checkpoint. The search order is:
1. Explicit `CHECKPOINT_PATH` (if set above)
2. `inference.checkpoint_path` from the YAML
3. `best.ckpt` or `last.ckpt` under the experiment directory

In [8]:
from merra_prism_inference import _find_checkpoint

checkpoint_path = _find_checkpoint(cfg, CHECKPOINT_PATH)
print(f"Using checkpoint: {checkpoint_path}")

Using checkpoint: /data/granite-wxc/examples/MERRA_PRISM/experiments/checkpoints/merra_prism_v1/best.ckpt


---

## Load Model

Recreate the model architecture and load the trained weights.

In [9]:
from merra_prism_inference import _load_model

model = _load_model(config, checkpoint_path, device)

total_params = sum(p.numel() for p in model.parameters())
print(f"Model loaded with {total_params:,} parameters")
print(f"Model is on: {next(model.parameters()).device}")

Creating the model.
[predictands] tmax: allow_negative_value=False but nonnegativity.enabled=False. This is expected for temperature-like variables.
[predictands] tmin: allow_negative_value=False but nonnegativity.enabled=False. This is expected for temperature-like variables.
[predictands] tmax: allow_negative_value=False but nonnegativity.enabled=False. This is expected for temperature-like variables.
[predictands] tmin: allow_negative_value=False but nonnegativity.enabled=False. This is expected for temperature-like variables.
--> model has 246,413,834 params.
Model loaded with 377,289,610 parameters
Model is on: cuda:0


---

## Run Inference

Execute inference over the date range defined in the YAML configuration.
The output is denormalized to physical units and written to a NetCDF file.

In [10]:
from merra_prism_inference import run_inference

output_dir = OUTPUT_DIR or str(
    resolve_path(
        cfg.get("inference", {}).get(
            "output_dir", "./examples/MERRA_PRISM/experiments/inference_output"
        )
    )
)

output_path = run_inference(
    cfg=cfg,
    config=config,
    checkpoint_path=checkpoint_path,
    output_dir=output_dir,
    device=device,
    batch_size=BATCH_SIZE,
)

print(f"\nInference output saved to: {output_path}")

[inference] loading checkpoint: /data/granite-wxc/examples/MERRA_PRISM/experiments/checkpoints/merra_prism_v1/best.ckpt
Creating the model.
[predictands] tmax: allow_negative_value=False but nonnegativity.enabled=False. This is expected for temperature-like variables.
[predictands] tmin: allow_negative_value=False but nonnegativity.enabled=False. This is expected for temperature-like variables.
[predictands] tmax: allow_negative_value=False but nonnegativity.enabled=False. This is expected for temperature-like variables.
[predictands] tmin: allow_negative_value=False but nonnegativity.enabled=False. This is expected for temperature-like variables.
--> model has 246,413,834 params.
[dataset] loaded static elevation (3105, 7025) from prism_elevation.nc
[inference] 3640 dates in inference period
[inference] case_name=merra_prism_v1
[inference] output_dir=/data/granite-wxc/examples/MERRA_PRISM/experiments/inference_output


MERRA-PRISM inference:   0%|          | 0/3640 [00:00<?, ?batch/s]

[inference] daily outputs saved → /data/granite-wxc/examples/MERRA_PRISM/experiments/inference_output

Inference output saved to: /data/granite-wxc/examples/MERRA_PRISM/experiments/inference_output


---

## Visualize Results

Load the output NetCDF and plot sample predictions for each target variable.

In [11]:
import xarray as xr

ds = xr.open_dataset(str(output_path))
print(ds)
print(f"\nTime steps: {len(ds.time)}")
print(f"Spatial grid: {len(ds.lat)} x {len(ds.lon)}")

ValueError: did not find a match in any of xarray's currently installed IO backends ['h5netcdf', 'scipy']. Consider explicitly selecting one of the installed engines via the ``engine`` parameter, or installing additional IO dependencies, see:
https://docs.xarray.dev/en/stable/getting-started-guide/installing.html
https://docs.xarray.dev/en/stable/user-guide/io.html

In [ ]:
import matplotlib.pyplot as plt

target_variables = cfg["data"]["target_variables"]
n_vars = len(target_variables)

fig, axes = plt.subplots(1, n_vars, figsize=(6 * n_vars, 5))
if n_vars == 1:
    axes = [axes]

# Plot the first time step for each variable
for ax, var in zip(axes, target_variables):
    data = ds[var].isel(time=0)
    im = ax.pcolormesh(ds.lon, ds.lat, data.values, shading="auto", cmap="coolwarm")
    ax.set_title(f"{var} (t=0)")
    ax.set_xlabel("Longitude")
    ax.set_ylabel("Latitude")
    plt.colorbar(im, ax=ax, shrink=0.8)

plt.suptitle("MERRA2-to-PRISM Downscaling Predictions", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

### Spatial Mean Time Series

Plot the spatial-mean time series for each variable to check temporal consistency.

In [ ]:
fig, axes = plt.subplots(n_vars, 1, figsize=(10, 4 * n_vars), sharex=True)
if n_vars == 1:
    axes = [axes]

preview_steps = min(7, len(ds.time))
preview_time = ds.time.isel(time=slice(0, preview_steps)).values

for ax, var in zip(axes, target_variables):
    spatial_mean = []
    for idx in range(preview_steps):
        spatial_mean.append(float(ds[var].isel(time=idx).mean(dim=["lat", "lon"]).values))
    ax.plot(preview_time, spatial_mean, linewidth=0.8, marker="o", markersize=3)
    ax.set_ylabel(var)
    ax.set_title(f"Spatial-mean {var}")
    ax.grid(True, alpha=0.3)

axes[-1].set_xlabel("Time")
plt.tight_layout()
plt.show()

In [ ]:
ds.close()
print("Done.")

---

## Summary

The inference pipeline:
1. Loaded the trained checkpoint from the experiment directory
2. Ran predictions over the inference date range
3. Denormalized outputs using pre-computed target scalars
4. Saved results as a NetCDF file with proper coordinates and metadata

Output file location:
```
examples/MERRA_PRISM/experiments/inference_output/merra_prism_inference.nc
```